In [10]:
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# ==========================================
# IBM QUANTUM CONNECTION
# ==========================================

API_KEY = "3rHiqWEZxxKhHKsxdtUF9DPokAZOctyadUnIsxI5oM-F"
CRN = "crn:v1:bluemix:public:quantum-computing:us-east:a/2f5af837dabb4d2481000377b323c1b6:691f5b63-c55d-43dd-b28c-ca3f89f398e0::"

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=API_KEY,
    instance=CRN
)

print("IBM Quantum connected successfully!")

qiskit_runtime_service._discover_account:WARNING:2026-10-10 12:27:19,452: Loading account with the given token. A saved account will not be used.


IBM Quantum connected successfully!


In [11]:
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=2
)

print("Using backend:", backend.name)

Using backend: ibm_fez


In [12]:
# ==========================================
# ORIGINAL BELL CIRCUIT
# ==========================================

bell = QuantumCircuit(2, 2)

bell.h(0)
bell.cx(0, 1)

bell.measure(0, 0)
bell.measure(1, 1)

print("Original Bell Circuit:")
print(bell)

Original Bell Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [13]:
# ==========================================
# MODIFIED BELL CIRCUIT
# ==========================================

modified = QuantumCircuit(2, 2)

modified.h(0)
modified.cx(0, 1)
modified.x(1)

modified.measure(0, 0)
modified.measure(1, 1)

print("Modified Circuit:")
print(modified)

Modified Circuit:
     ┌───┐          ┌─┐   
q_0: ┤ H ├──■───────┤M├───
     └───┘┌─┴─┐┌───┐└╥┘┌─┐
q_1: ─────┤ X ├┤ X ├─╫─┤M├
          └───┘└───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 


In [14]:
# ==========================================
# TRANSPILE
# ==========================================

bell_transpiled = transpile(bell, backend)
modified_transpiled = transpile(modified, backend)

print("Both circuits transpiled successfully!")

Both circuits transpiled successfully!


In [15]:
# ==========================================
# RUN ON IBM QUANTUM HARDWARE
# ==========================================

sampler = SamplerV2(mode=backend)

job = sampler.run(
    [bell_transpiled, modified_transpiled],
    shots=100
)

print("Job submitted!")
print("Job ID:", job.job_id())

C:\Users\bhavi\AppData\Local\Temp\ipykernel_23948\1333199449.py:5: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = SamplerV2(mode=backend)


Job submitted!
Job ID: db4u4i2mb58s7389o3n0


In [16]:
result = job.result()

print("Job completed!")

Job completed!


In [17]:
# ==========================================
# GET COUNTS
# ==========================================

original_counts = result[0].data.c.get_counts()
modified_counts = result[1].data.c.get_counts()

print("Original Bell Circuit:")
print(original_counts)

print("\nModified Circuit:")
print(modified_counts)

Original Bell Circuit:
{'00': 56, '11': 42, '10': 1, '01': 1}

Modified Circuit:
{'01': 54, '10': 46}


In [18]:
# ==========================================
# HISTOGRAMS
# ==========================================

plot_histogram(
    [original_counts, modified_counts],
    legend=[
        "Original Bell State",
        "Modified Circuit"
    ]
)

plt.show()